# GBV Study Data Classification Notebook

## Project Overview
This notebook processes data from a Gender-Based Violence (GBV) study report, aiming to classify different sections of the report based on their textual and numerical content. The primary goal is to build a machine learning model that can predict the 'section' (e.g., 'PREVALENCE', 'INJURIES', 'RECOMMENDATIONS') of an entry within the report.

## Data Source
The data is loaded from a CSV file: `/content/drive/MyDrive/full-report-the-first-south-african-national-gender-based-violence-study-2022.txt`.

## Methodology
The process involves several key stages:

1.  **Data Loading and Initial Inspection**: The raw data is loaded into a Pandas DataFrame, and initial checks are performed to understand its structure, data types, and missing values.

2.  **Data Preprocessing and Cleaning**:
    *   Missing `notes` values are filled with empty strings.
    *   A `value_numeric` column is created by extracting numerical information from the `value` column, handling percentages and text-embedded numbers (e.g., "7,310,389 women"). Non-convertible values are coerced to `NaN`.
    *   A combined `text` column is generated from `indicator` and `notes` for natural language processing.

3.  **Feature Engineering**:
    *   The `section` column is transformed into a numerical target variable (`y_encoded`) using `LabelEncoder`.
    *   Textual features (`X_text`) are created using `TfidfVectorizer`.
    *   Numerical features (`X_numeric_scaled`) are derived from `value_numeric`, with `SimpleImputer` handling `NaN`s (imputing with 0) and `StandardScaler` standardizing the values.
    *   All features are combined into a single feature matrix `X` using `hstack`. Robust type conversion and a final `SimpleImputer` step ensure `X` is a dense, NaN-free `float` NumPy array.

4.  **Data Splitting**: The data is split into training (70%), validation (15%), and testing (15%) sets. Special handling is implemented to address rare classes during splitting to avoid issues with `stratify` parameters.

5.  **Model Training**:
    *   A `RandomForestClassifier` is initialized and trained on the `X_train` and `y_train` data.
    *   Extensive debug checks were added during the process to ensure `X_train` is free of `NaN`s and is of the correct data type (`float` NumPy array) before model fitting.

## Current Status
*   Data has been successfully loaded, cleaned, and features engineered.
*   The data has been split into training, validation, and test sets.
*   A `RandomForestClassifier` model has been trained on the processed training data without encountering `NaN` errors.
*   The cleaned DataFrame `df_clean` has been saved to `gbv_data_cleaned.csv`.

In [36]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')


file_path = "/content/drive/MyDrive/full-report-the-first-south-african-national-gender-based-violence-study-2022.txt"

In [21]:
df = pd.read_csv(file_path)

In [22]:
df.head()

,section,category,subcategory,indicator,value,value_type,notes
0,STUDY_OVERVIEW,Study_Info,Title,The First South African National Gender-Based ...,text,NaN,NaN
1,STUDY_OVERVIEW,Study_Info,Subtitle,A Baseline Survey on Victimisation and Perpetr...,text,NaN,NaN
2,STUDY_OVERVIEW,Study_Info,Publisher,Human Sciences Research Council,text,NaN,NaN
3,STUDY_OVERVIEW,Study_Info,Publication_Year,2024,text,NaN,NaN
4,STUDY_OVERVIEW,Study_Info,ISBN_Print,978-1-0370-2839-7,text,NaN,NaN


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 231 entries, 0 to 230
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   section      231 non-null    object 
 1   category     231 non-null    object 
 2   subcategory  231 non-null    object 
 3   indicator    231 non-null    object 
 4   value        231 non-null    object 
 5   value_type   37 non-null     object 
 6   notes        0 non-null      float64
dtypes: float64(1), object(6)
memory usage: 12.8+ KB


In [24]:
# 3. Check for Missing Values per Column, the extect number and percentage of the missing values
print("--- MISSING VALUE COUNT ---")
missing_info = pd.DataFrame({
    'Missing Values': df.isnull().sum(),
    'Percentage (%)': (df.isnull().sum() / len(df)) * 100
})
print(missing_info)

--- MISSING VALUE COUNT ---
             Missing Values  Percentage (%)
section                   0        0.000000
category                  0        0.000000
subcategory               0        0.000000
indicator                 0        0.000000
value                     0        0.000000
value_type              194       83.982684
notes                   231      100.000000


In [27]:
# STEP 2: UNDERSTAND THE DATA
# ============================================
print("\n" + "-" * 60)
print("STEP 2: Understanding the Dataset")
print("-" * 60)

print("\n📊 Dataset Contents:")
print(f"   - Number of sections: {df['section'].nunique()}")
print(f"   - Sections: {df['section'].unique().tolist()}")

print("\n📊 Data Types:")
print(df.dtypes)

print("\n📊 Missing Values:")
print(df.isnull().sum())

# Count rows per section
print("\n📊 Rows per section:")
for section in df['section'].value_counts().index:
    count = df[df['section'] == section].shape[0]
    print(f"   - {section}: {count} rows")


------------------------------------------------------------
STEP 2: Understanding the Dataset
------------------------------------------------------------

📊 Dataset Contents:
   - Number of sections: 13
   - Sections: ['STUDY_OVERVIEW', 'PREVALENCE', 'VULNERABLE', 'FACTORS', 'EMOTIONAL_ABUSE', 'ECONOMIC_ABUSE', 'CONTROLLING', 'INJURIES', 'HELP_SEEKING', 'GENDER_NORMS', 'COVID', 'LAWS', 'RECOMMENDATIONS']

📊 Data Types:
section         object
category        object
subcategory     object
indicator       object
value           object
value_type      object
notes          float64
dtype: object

📊 Missing Values:
section          0
category         0
subcategory      0
indicator        0
value            0
value_type     194
notes          231
dtype: int64

📊 Rows per section:
   - STUDY_OVERVIEW: 53 rows
   - PREVALENCE: 34 rows
   - VULNERABLE: 22 rows
   - GENDER_NORMS: 22 rows
   - RECOMMENDATIONS: 22 rows
   - FACTORS: 21 rows
   - COVID: 11 rows
   - EMOTIONAL_ABUSE: 9 rows
   - C

In [44]:
# STEP 3: PREPROCESS THE DATA
# ============================================
print("\n" + "-" * 60)
print("STEP 3: Data Preprocessing")
print("-" * 60)

# Create a copy of the data
df_clean = df.copy()

# 3.1: Handle missing values
print("\n3.1: Handling missing values...")
df_clean['notes'] = df_clean['notes'].fillna('')
print(f"   ✅ Filled {df['notes'].isna().sum()} missing values")

# 3.2: Create numeric column from value
print("\n3.2: Creating numeric column...")

def convert_to_number(row):
    """Convert value to number"""
    val = row['value']
    val_type = row['value_type']

    if isinstance(val, (int, float)):
        return val
    if pd.isna(val) or str(val).strip() == 'NaN':
        return np.nan

    cleaned_val = str(val).strip().replace(',', '')

    # Handle percentages
    if cleaned_val.endswith('%'):
        try:
            return float(cleaned_val.replace('%', '')) / 100
        except ValueError:
            pass

    # Handle values with 'women' or 'men' suffix
    if 'women' in cleaned_val or 'men' in cleaned_val:
        try:
            # Extract leading number before 'women' or 'men'
            num_str = cleaned_val.split(' ')[0]
            return float(num_str)
        except ValueError:
            pass

    # Attempt general float conversion
    try:
        return float(cleaned_val)
    except ValueError:
        return np.nan

df_clean['value_numeric'] = df_clean.apply(convert_to_number, axis=1)
print(f"   ✅ Created 'value_numeric' column. Sample: {df_clean['value_numeric'].head().tolist()}")


------------------------------------------------------------
STEP 3: Data Preprocessing
------------------------------------------------------------

3.1: Handling missing values...
   ✅ Filled 231 missing values

3.2: Creating numeric column...
   ✅ Created 'value_numeric' column. Sample: [nan, nan, nan, nan, nan]


In [29]:
# 3.3: Create text column for NLP
print("\n3.3: Creating text column...")
df_clean['text'] = df_clean['indicator'] + ' ' + df_clean['notes']
print(f"   ✅ Created text column")
print(f"   Sample: {df_clean['text'].iloc[0][:60]}...")


3.3: Creating text column...
   ✅ Created text column
   Sample: The First South African National Gender-Based Violence Study...


In [30]:
# 3.4: Create target variable
print("\n3.4: Creating target variable...")
# Use 'section' as target (what we want to predict)
y = df_clean['section']
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"   ✅ Target created with {len(le.classes_)} classes")
print(f"   Classes: {le.classes_.tolist()}")



3.4: Creating target variable...
   ✅ Target created with 13 classes
   Classes: ['CONTROLLING', 'COVID', 'ECONOMIC_ABUSE', 'EMOTIONAL_ABUSE', 'FACTORS', 'GENDER_NORMS', 'HELP_SEEKING', 'INJURIES', 'LAWS', 'PREVALENCE', 'RECOMMENDATIONS', 'STUDY_OVERVIEW', 'VULNERABLE']


In [33]:
# 3.5: Create features
print("\n3.5: Creating features...")



3.5: Creating features...


In [34]:
# Text feature (TF-IDF)
vectorizer = TfidfVectorizer(max_features=50)
X_text = vectorizer.fit_transform(df_clean['text'])

In [40]:
# Further inspect why 'value_numeric' is all NaNs
print("\n🔍 Inspecting 'value' entries that resulted in NaN in 'value_numeric':")
non_numeric_values = df_clean[df_clean['value_numeric'].isna()]['value'].unique()
print(f"Total unique non-convertible 'value' entries: {len(non_numeric_values)}")
if len(non_numeric_values) > 10:
    print("Top 10 samples of non-convertible 'value' entries:")
    for val in non_numeric_values[:10]:
        print(f"   - {val}")
else:
    print("All non-convertible 'value' entries:")
    for val in non_numeric_values:
        print(f"   - {val}")

# Also check value_type distribution for these non-convertible values
print("\nDistribution of 'value_type' for non-convertible 'value' entries:")
print(df_clean[df_clean['value_numeric'].isna()]['value_type'].value_counts(dropna=False))


🔍 Inspecting 'value' entries that resulted in NaN in 'value_numeric':
Total unique non-convertible 'value' entries: 30
Top 10 samples of non-convertible 'value' entries:
   - text
   - numeric
   - percentage
   - Significantly higher among married men
   - Significantly higher with tertiary education
   - Significantly higher among employed men
   - Significantly higher in urban areas
   - Significantly higher agreement among Black African men
   - Review mental health services for GBV survivors, children who witnessed GBV, and men
   - Integrate SRH&R services with GBV services for early detection

Distribution of 'value_type' for non-convertible 'value' entries:
value_type
NaN                  194
With disability        9
7,310,389 women        1
2,150,342 women        1
7,847,438 women        1
Small Area Layers      1
432,525 women          1
1,536,729 women        1
3,221,649 women        1
1,131,293 women        1
3,448,669 women        1
747,188 women          1
354,196 women 

In [54]:
# Combine features
from scipy.sparse import hstack
import numpy as np # Ensure numpy is imported for nan check

# 3.6: Scale numeric features (Moved from xk8nc0boNfHD)
print("\n3.6: Scaling and Imputing numeric features...")

# Impute missing values in value_numeric BEFORE scaling
# Use 'constant' strategy with fill_value=0 to ensure no NaNs, even if all original values are NaN
imputer = SimpleImputer(strategy='constant', fill_value=0)
df_clean['value_numeric_imputed'] = imputer.fit_transform(df_clean[['value_numeric']])

scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(df_clean[['value_numeric_imputed']])
print(f"   ✅ Numeric features imputed and scaled")

# X_text is sparse, X_numeric_scaled is dense. hstack will result in sparse.
X = hstack([X_text, X_numeric_scaled])

# Explicitly convert to dense array BEFORE final imputation to ensure all elements are visible
# and SimpleImputer processes a dense structure, guaranteeing a dense output.
if hasattr(X, "toarray"):
    X = X.toarray()

# Impute any remaining NaNs in the combined feature matrix X
# This step should now operate on a dense array and return a dense array.
imputer_final = SimpleImputer(strategy='constant', fill_value=0)
X = imputer_final.fit_transform(X)

# Final check for NaNs after all imputation
if np.isnan(X).any():
    print("   ❌ WARNING: NaNs STILL present in X after all processing!")
else:
    print("   ✅ X is completely free of NaNs.")

print(f"   ✅ Features created and imputed! Shape: {X.shape}")
print(f"   Type of X after final processing: {type(X)}")


3.6: Scaling and Imputing numeric features...
   ✅ Numeric features imputed and scaled
   ✅ X is completely free of NaNs.
   ✅ Features created and imputed! Shape: (231, 51)
   Type of X after final processing: <class 'numpy.ndarray'>


In [45]:
# 3.7: Split data
print("\n3.7: Splitting data (Train/Validation/Test)...")

# Split: 70% train, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_encoded, test_size=0.30, random_state=42, stratify=y_encoded
)

# Identify classes in y_temp that have only one sample
unique_classes_in_ytemp, counts_in_ytemp = np.unique(y_temp, return_counts=True)
problematic_classes_indices = np.where(counts_in_ytemp < 2)[0]

if len(problematic_classes_indices) > 0:
    problematic_classes = unique_classes_in_ytemp[problematic_classes_indices]
    print(f"   ⚠️ Warning: The following classes have <2 samples in y_temp and will be excluded from Val/Test splits: {le.inverse_transform(problematic_classes).tolist()}")

    # Filter X_temp and y_temp to remove these problematic samples
    mask = ~np.isin(y_temp, problematic_classes)
    X_temp_filtered = X_temp[mask]
    y_temp_filtered = y_temp[mask]
else:
    X_temp_filtered = X_temp
    y_temp_filtered = y_temp


# Perform the second split on the filtered temporary sets
X_val, X_test, y_val, y_test = train_test_split(
    X_temp_filtered, y_temp_filtered, test_size=0.50, random_state=42, stratify=y_temp_filtered
)

print(f"   ✅ Data split complete!")
print(f"   Training: {X_train.shape[0]} samples")
print(f"   Validation: {X_val.shape[0]} samples")
print(f"   Test: {X_test.shape[0]} samples")


3.7: Splitting data (Train/Validation/Test)...
   ⚠️ Warning: The following classes have <2 samples in y_temp and will be excluded from Val/Test splits: ['INJURIES']
   ✅ Data split complete!
   Training: 161 samples
   Validation: 34 samples
   Test: 35 samples


In [47]:
#TRAIN A SIMPLE MODEL (DEMO)
# ============================================
print("\n" + "-" * 60)
print("Training a Simple Model (Demo)")
print("-" * 60)


------------------------------------------------------------
Training a Simple Model (Demo)
------------------------------------------------------------


In [57]:
# Random Forest classifier
import numpy as np
from sklearn.impute import SimpleImputer
import pandas as pd # Ensure pandas is imported for to_numeric

print(f"DEBUG (vhiQlVlOOWw8): --- Entry point --- ")
print(f"DEBUG (vhiQlVlOOWw8): X_train type: {type(X_train)}")
print(f"DEBUG (vhiQlVlOOWw8): X_train dtype: {getattr(X_train, 'dtype', 'No dtype attribute')}")
print(f"DEBUG (vhiQlVlOOWw8): X_train shape: {getattr(X_train, 'shape', 'No shape attribute')}")
if hasattr(X_train, 'shape') and X_train.shape[0] > 0 and X_train.shape[1] > 0:
    print(f"DEBUG (vhiQlVlOOWw8): X_train[:2, :5] (sample):\n{X_train[:2, :5]}")
else:
    print(f"DEBUG (vhiQlVlOOWw8): X_train is empty or malformed for sample printing.")

# Force X_train to be a dense float numpy array, coercing errors
try:
    # If it's a sparse matrix, convert to dense array first
    if hasattr(X_train, 'toarray'):
        print("DEBUG (vhiQlVlOOWw8): X_train detected as sparse, converting to dense.")
        X_train_processed = X_train.toarray()
    else:
        X_train_processed = X_train # Assume it's already dense or can be treated as such

    # Then convert to float, coercing any non-numeric values to NaN using pd.to_numeric
    print(f"DEBUG (vhiQlVlOOWw8): Attempting to convert X_train to float dtype using pd.to_numeric with errors='coerce'.")
    # Apply to_numeric column-wise or flatten for 1D, then reshape
    # It's safest to convert to a DataFrame temporarily if X_train_processed is 2D
    if X_train_processed.ndim == 2:
        X_train_processed = pd.DataFrame(X_train_processed).apply(pd.to_numeric, errors='coerce').values
    else:
        X_train_processed = pd.to_numeric(X_train_processed, errors='coerce').values

    # Check if any NaNs were introduced by coercion (i.e., non-numeric values were present)
    if np.isnan(X_train_processed).any():
        print("DEBUG (vhiQlVlOOWw8): NaNs introduced during float conversion (errors='coerce'). Imputing these NaNs...")
        imputer_coerce = SimpleImputer(strategy='constant', fill_value=0)
        X_train = imputer_coerce.fit_transform(X_train_processed)
        print("DEBUG (vhiQlVlOOWw8): Coerced NaNs imputed with 0.")
    else:
        X_train = X_train_processed
        print("DEBUG (vhiQlVlOOWw8): Float conversion successful, no NaNs introduced by coercion.")

except Exception as e:
    print(f"DEBUG (vhiQlVlOOWw8): CRITICAL ERROR during forced conversion of X_train: {e}")
    print(f"DEBUG (vhiQlVlOOWw8): X_train type (before failure): {type(X_train)}")
    print(f"DEBUG (vhiQlVlOOWw8): X_train dtype (before failure): {getattr(X_train, 'dtype', 'No dtype attribute')}")
    raise # Re-raise the exception to stop execution and prevent misleading results

print(f"DEBUG (vhiQlVlOOWw8): --- After aggressive conversion --- ")
print(f"DEBUG (vhiQlVlOOWw8): X_train type: {type(X_train)}")
print(f"DEBUG (vhiQlVlOOWw8): X_train dtype: {X_train.dtype}")
print(f"DEBUG (vhiQlVlOOWw8): X_train shape: {X_train.shape}")
if X_train.shape[0] > 0 and X_train.shape[1] > 0:
    print(f"DEBUG (vhiQlVlOOWw8): X_train[:2, :5] (sample):\n{X_train[:2, :5]}")

# Final NaN check before fitting the model (this check should now pass without TypeError)
if np.isnan(X_train).any():
    print("\n❌ ERROR: X_train still contains NaN values before model fit! (This should not happen now)")
    # This block is a fallback, but ideally should not be reached if previous steps worked.
    imputer_final_train = SimpleImputer(strategy='constant', fill_value=0)
    X_train = imputer_final_train.fit_transform(X_train)
    print("   Further NaNs in X_train imputed with 0 for safety.")
else:
    print("\n✅ X_train is NaN-free. Proceeding with model training.")

model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)

DEBUG (vhiQlVlOOWw8): --- Entry point --- 
DEBUG (vhiQlVlOOWw8): X_train type: <class 'scipy.sparse._csr.csr_matrix'>
DEBUG (vhiQlVlOOWw8): X_train dtype: float64
DEBUG (vhiQlVlOOWw8): X_train shape: (161, 51)
DEBUG (vhiQlVlOOWw8): X_train[:2, :5] (sample):
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1 stored elements and shape (2, 5)>
  Coords	Values
  (1, 3)	1.0
DEBUG (vhiQlVlOOWw8): X_train detected as sparse, converting to dense.
DEBUG (vhiQlVlOOWw8): Attempting to convert X_train to float dtype using pd.to_numeric with errors='coerce'.
DEBUG (vhiQlVlOOWw8): NaNs introduced during float conversion (errors='coerce'). Imputing these NaNs...
DEBUG (vhiQlVlOOWw8): Coerced NaNs imputed with 0.
DEBUG (vhiQlVlOOWw8): --- After aggressive conversion --- 
DEBUG (vhiQlVlOOWw8): X_train type: <class 'numpy.ndarray'>
DEBUG (vhiQlVlOOWw8): X_train dtype: float64
DEBUG (vhiQlVlOOWw8): X_train shape: (161, 51)
DEBUG (vhiQlVlOOWw8): X_train[:2, :5] (sample):
[[0. 0. 0. 0. 0.]
 [0

RandomForestClassifier(n_estimators=50, random_state=42)

In [58]:
# Save cleaned data
df_clean.to_csv('gbv_data_cleaned.csv', index=False)
print("✅ Saved: gbv_data_cleaned.csv")


✅ Saved: gbv_data_cleaned.csv
